In [0]:
# Notebook 03: 03_anomaly_detection.ipynb
# Etapa 4: Treinamento e Avaliação de Detecção de Anomalias
# Abortamos a ideia de treinar o Modelo de Guassian Mixture Model (GMM) por causa da sua ineficácia em detectar anomalias.
# A ideia era usar o modelo não supervisionado GMM para detectar anomalias no dataset, mas a hiperdimensionalidade do dataset tornou o treinamento do modelo ineficiente.
# Ainda assim, vamos manter o código comentado para futuras referências.

#from pyspark.ml.clustering import GaussianMixture
#from pyspark.ml.evaluation import MulticlassClassificationEvaluator
#from pyspark.sql.functions import col, count, lit

# --- 1. Recuperar o DataFrame Vetorizado ---

#TABLE_NAME = "bigdata_anomaly_detection_kddcup99_catalogue.default.kdd_vector_features_pca"

#try:
#    df_vector = spark.table(TABLE_NAME)
#    print(f"Sucesso! DataFrame vetorizado carregado da Tabela Delta: {TABLE_NAME}")
#    print(f"Total de registros: {df_vector.count():,}")
#    df_vector.printSchema()

#except Exception as e:
#    print(f"ERRO: Não foi possível carregar a Tabela Delta {TABLE_NAME}.")
#    print("Certifique-se que o Notebook 02 foi executado e salvou a tabela corretamente.")
    # raise e

# --- 2. Treinamento do GMM (Distribuído) ---

# Define o modelo GMM (k=2: Cluster Normal vs. Cluster Anomalia)
#gmm = GaussianMixture(
#    k=2, 
#    maxIter=50, 
#    featuresCol="features",
#    predictionCol="prediction_cluster"
#)

#print("\nIniciando treinamento do Gaussian Mixture Model (k=2) de forma distribuída...")
# O .fit() é processado em paralelo pelos workers do Spark
#gmm_model = gmm.fit(df_vector)

#print("Treinamento concluído.")


# --- 3. Inferência e Mapeamento dos Clusters ---

# 3.1. Fazer Previsões (Inferência no dataset completo)
#df_predictions = gmm_model.transform(df_vector)

# 3.2. Análise da Distribuição dos Clusters
#print("\n--- Distribuição dos Clusters Preditos ---")
#df_predictions.groupBy("prediction_cluster").count().show()

# 3.3. Mapeamento: Cruzamento da Classe Verdadeira (is_anomaly) com o Cluster Predito
#print("\n--- Distribuição da Classe Verdadeira (is_anomaly) dentro dos Clusters Preditos ---")
#print("is_anomaly: 0=Normal, 1=Anomalia (Intrusão)")

# Essa tabela revela se o GMM conseguiu isolar as duas classes
#df_predictions.groupBy("prediction_cluster", "is_anomaly").agg(
#    count(lit(1)).alias("count")
#).orderBy(col("prediction_cluster"), col("is_anomaly")).show()

In [0]:
# Notebook 03: 03_random_forest_classification.ipynb
# Etapa 5: Treinamento e Avaliação do Random Forest (Supervisionado)

from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.functions import col, count, lit, sum as spark_sum, when

# --- 1. Configuração e Carregamento dos Dados de Treino e Teste (Processados) ---

BASE_TABLE_NAME = "bigdata_anomaly_detection_kddcup99_catalogue.default"
TRAIN_TABLE_NAME = f"{BASE_TABLE_NAME}.kdd_features_train"
TEST_TABLE_NAME = f"{BASE_TABLE_NAME}.kdd_features_test"

try:
    # Carrega o conjunto de TREINO (Escalado + PCA, sem vazamento)
    train_data = spark.table(TRAIN_TABLE_NAME)
    # Carrega o conjunto de TESTE (Escalado + PCA, sem vazamento)
    test_data = spark.table(TEST_TABLE_NAME)

    print(f"Sucesso! Dados de Treino e Teste (PCA) carregados do UC.")
    print(f"Dados de Treinamento: {train_data.count():,} registros")
    print(f"Dados de Teste: {test_data.count():,} registros")

    # Confirma o schema (deve ser: features: vectorudt, is_anomaly: integer)
    train_data.printSchema()

except Exception as e:
    print("ERRO: Não foi possível carregar as Tabelas Delta. Verifique os nomes e permissões.")
    raise e


# --- 2. Treinamento do Modelo Random Forest ---

# Define o classificador Random Forest
# Usamos os parâmetros que você definiu: numTrees=20, maxDepth=5
rf = RandomForestClassifier(
    labelCol="is_anomaly", 
    featuresCol="features", 
    numTrees=20, 
    maxDepth=5, 
    seed=42
)

print("\nIniciando treinamento do Random Forest Classifier de forma distribuída...")
# O treinamento é feito no conjunto de treino carregado
rf_model = rf.fit(train_data)

print("Treinamento concluído.")


# --- 3. Inferência e Avaliação do Modelo ---

# 3.1. Fazer Previsões (Inferência no conjunto de Teste)
df_predictions = rf_model.transform(test_data)
df_predictions = df_predictions.withColumnRenamed("prediction", "predicted_anomaly")

print("\n--- Distribuição das Previsões no Conjunto de Teste ---")
df_predictions.groupBy("predicted_anomaly").count().show()


# 3.2. Avaliação de Performance (Métricas)

# A. Área Sob a Curva ROC (AUC-ROC)
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="is_anomaly",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = evaluator_auc.evaluate(df_predictions)
print(f"\n✅ AUC-ROC no Conjunto de Teste: {auc:.4f}")

# B. Matriz de Confusão
print("\n--- Matriz de Confusão e Outras Métricas (Acurácia, F1) ---")

# 1. Geração da Matriz de Confusão (Usando DataFrame e Pivot)
confusion_matrix_df = df_predictions.groupBy("is_anomaly").pivot("predicted_anomaly", [0.0, 1.0]).agg(spark_sum(lit(1)))
confusion_matrix_df = confusion_matrix_df.fillna(0)

print("\nMatriz de Confusão (Linhas=Real, Colunas=Previsto):")
print("  | Previsto 0.0 (Normal) | Previsto 1.0 (Anomalia)")
confusion_matrix_df.show()


# 2. Extração e Cálculo de Métricas
# (O código de extração manual é mantido, mas verifique se a ordem das linhas na matriz está correta)

# Coleta a matriz para manipulação em Python
matrix_collected = confusion_matrix_df.collect()

# Garante que a classe real (is_anomaly) esteja na ordem [0.0, 1.0] para TP/TN/FP/FN
row_0 = matrix_collected[0] if matrix_collected[0]["is_anomaly"] == 0.0 else matrix_collected[1]
row_1 = matrix_collected[1] if matrix_collected[1]["is_anomaly"] == 1.0 else matrix_collected[0]

# TN (True Negative - Real 0, Previsto 0)
TN = row_0["0.0"]
# FP (False Positive - Real 0, Previsto 1)
FP = row_0["1.0"]
# FN (False Negative - Real 1, Previsto 0)
FN = row_1["0.0"]
# TP (True Positive - Real 1, Previsto 1)
TP = row_1["1.0"]

Total = TN + FP + FN + TP

# 3. Cálculo de Métricas
precision_denominator = (TP + FP) if (TP + FP) > 0 else 1
recall_denominator = (TP + FN) if (TP + FN) > 0 else 1

accuracy = (TP + TN) / Total
precision = TP / precision_denominator
recall = TP / recall_denominator
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"\nAcurácia: {accuracy:.4f}")
print(f"Precisão (Classe Anomalia '1'): {precision:.4f}")
print(f"Recall (Sensibilidade - Classe Anomalia '1'): {recall:.4f}")
print(f"F1-Score (Classe Anomalia '1'): {f1_score:.4f}")

# Exibe as métricas por classe para maior detalhe (Usando PySpark ML Evaluator)
evaluator_ml = MulticlassClassificationEvaluator(
    labelCol="is_anomaly", 
    predictionCol="predicted_anomaly", 
    metricName="f1"
)

f1_ml = evaluator_ml.evaluate(df_predictions)
accuracy_ml = evaluator_ml.evaluate(df_predictions, {evaluator_ml.metricName: "accuracy"})

print(f"\nVerificação (ML Evaluator):")
print(f"Acurácia (ML Evaluator): {accuracy_ml:.4f}")
print(f"F1-Score (ML Evaluator): {f1_ml:.4f}")